In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

# Paths — using your existing Volume
source_path     = "/Volumes/retail_q/volumes/blob_source/transactions_source/"
target_table    = "retail_q.blob_bronze.transaction"
checkpoint_path = "/Volumes/retail_q/volumes/blob_source/_checkpoints/transactions/stream"
schema_path     = "/Volumes/retail_q/volumes/blob_source/_checkpoints/transactions/schema"

# Explicit schema
transactions_schema = StructType([
    StructField("transaction_id",        StringType(),  nullable=True),
    StructField("opportunity_name",      StringType(),  nullable=True),
    StructField("product_id",            StringType(),  nullable=True),
    StructField("store_id",              StringType(),  nullable=True),
    StructField("quantity",              IntegerType(), nullable=True),
    StructField("selling_price",         DoubleType(),  nullable=True),
    StructField("discount_amount",       DoubleType(),  nullable=True),
    StructField("transaction_timestamp", StringType(),  nullable=True),
    StructField("payment_mode",          StringType(),  nullable=True),
    StructField("sales_channel",         StringType(),  nullable=True),
])

# Read
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .schema(transactions_schema)
    .load(source_path)
)

# Audit columns
df = (
    df
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file",         F.col("_metadata.file_path"))
)

# Write
(
    df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

In [0]:
%sql
SELECT COUNT(*) FROM retail_q.blob_bronze.transaction